# 01 — Exploratory Data Analysis and Baseline Model
**Dataset:** Bank Account Fraud (BAF) Suite, Base variant (Jesus et al., 2022)

This notebook runs unchanged on the MacBook, the university PC and Google Colab.

**Where to put the data**
- **Mac / university PC:** put `Base.csv` in a folder called `data` inside the repository (e.g. `fraude_disseratation/data/Base.csv`). The notebook searches the current folder and its parent folders for it. The `data/` folder is excluded from Git by `.gitignore`, so copy it to each machine once.
- **Google Colab:** upload `Base.csv` to Google Drive and edit `COLAB_DATA_DIR` in the first code cell.
- **Anywhere else:** set the environment variable `DISSERTATION_DATA_DIR` to the folder containing `Base.csv`.

On the first run the CSV is converted to `Base.parquet`, which loads much faster afterwards.

In [ ]:
import os
import sys
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

# ---- Work out where the data is, depending on which machine this is running on ----
COLAB_DATA_DIR = '/content/drive/MyDrive/Dissertation/data'   # edit to match your Drive folder

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path(COLAB_DATA_DIR)
    MACHINE = 'Google Colab'
elif os.environ.get('DISSERTATION_DATA_DIR'):
    DATA_DIR = Path(os.environ['DISSERTATION_DATA_DIR'])
    MACHINE = 'Local (DISSERTATION_DATA_DIR)'
else:
    # Search the current folder and every parent folder for data/Base.csv or data/Base.parquet
    here = Path.cwd().resolve()
    DATA_DIR = next(
        (p / 'data' for p in [here, *here.parents]
         if (p / 'data' / 'Base.csv').exists() or (p / 'data' / 'Base.parquet').exists()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError(
            "Could not find data/Base.csv. Put it in a 'data' folder inside the repository, "
            "or set the DISSERTATION_DATA_DIR environment variable."
        )
    MACHINE = 'Local'

OUTPUT_DIR = DATA_DIR.parent / 'outputs'
(OUTPUT_DIR / 'processed').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'results').mkdir(parents=True, exist_ok=True)

print(f"Machine:    {MACHINE}")
print(f"Data dir:   {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"pandas {pd.__version__}")

In [ ]:
def reduce_memory(df):
    """Shrink numeric types and store text columns as 'category' (roughly halves memory use)."""
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].astype('category')
    return df

csv_path = DATA_DIR / 'Base.csv'
parquet_path = DATA_DIR / 'Base.parquet'

# Load the fast Parquet copy if it exists, otherwise read the CSV once and save a Parquet copy
if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    df = reduce_memory(pd.read_csv(csv_path))
    df.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet copy to {parquet_path}")

df = reduce_memory(df)  # also makes sure the types are right if the Parquet file was made elsewhere

print("Shape:", df.shape)
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print("\nFirst 5 rows:")
df.head()

## Data quality
`isnull()` only finds true blanks. In BAF, missing values in several columns are encoded as **negative numbers** instead (see the BAF datasheet), so they are counted separately below.

In [ ]:
print("Shape:", df.shape)
print("\nMissing values (true blanks):")
print(df.isnull().sum())

In [ ]:
# Columns where a negative value means "missing" according to the BAF datasheet
MISSING_IF_NEGATIVE = [
    'prev_address_months_count',
    'current_address_months_count',
    'intended_balcon_amount',
    'bank_months_count',
    'session_length_in_minutes',
    'device_distinct_emails_8w',
]

hidden_missing = pd.DataFrame({
    'n_missing': {c: int((df[c] < 0).sum()) for c in MISSING_IF_NEGATIVE if c in df.columns},
    'pct_missing': {c: round((df[c] < 0).mean() * 100, 2) for c in MISSING_IF_NEGATIVE if c in df.columns},
}).sort_values('pct_missing', ascending=False)

print("Hidden missing values (encoded as negative numbers):")
hidden_missing

# Note: the values are left as they are. Tree models (LightGBM/XGBoost) can use "missing" as a signal.
# For logistic regression this is a modelling choice worth discussing in the methodology chapter.

In [ ]:
# Drop columns that never change (they carry no information, e.g. device_fraud_count in the Base variant)
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=constant_cols)
print("Dropped constant columns:", constant_cols)
print("New shape:", df.shape)

## Class distribution and summary statistics

In [ ]:
counts = df['fraud_bool'].value_counts()
print(counts)
print("\nFraud percentage:", round(df['fraud_bool'].mean() * 100, 2), "%")

# Plot
df['fraud_bool'].value_counts().sort_index().plot(kind='bar', color=['steelblue', 'tomato'])
plt.xticks([0, 1], ['Legitimate', 'Fraud'], rotation=0)
plt.title('Class Distribution')
plt.ylabel('Count')
plt.show()

In [ ]:
df.describe()

In [ ]:
# Separate numerical and categorical columns
# (text columns are stored as 'category' by reduce_memory; this line works in pandas 2 and 3)
categorical_cols = df.select_dtypes(include=['category', 'object', 'string']).columns.tolist()
numerical_cols = df.select_dtypes(include='number').columns.tolist()
numerical_cols.remove('fraud_bool')  # remove target variable

print("Categorical columns:", categorical_cols)
print("\nNumerical columns:", numerical_cols)

In [ ]:
for col in categorical_cols:
    print(f"{col}: {list(df[col].unique())}")

## Age and categorical features vs fraud

In [ ]:
plt.figure(figsize=(10, 5))
df[df['fraud_bool'] == 0]['customer_age'].hist(alpha=0.6, bins=20, label='Legitimate', color='steelblue')
df[df['fraud_bool'] == 1]['customer_age'].hist(alpha=0.6, bins=20, label='Fraud', color='tomato')
plt.xlabel('Customer Age')
plt.ylabel('Count')
plt.title('Age Distribution: Fraud vs Legitimate')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Create age groups
df['age_group'] = pd.cut(df['customer_age'],
                          bins=[0, 20, 30, 40, 50, 60, 100],
                          labels=['<20', '20-30', '30-40', '40-50', '50-60', '60+'])

# Fraud rate per age group
fraud_by_age = df.groupby('age_group', observed=True)['fraud_bool'].mean() * 100

print("Fraud rate by age group (%):")
print(fraud_by_age.round(2))

# Plot
fraud_by_age.plot(kind='bar', color='tomato', edgecolor='black')
plt.title('Fraud Rate by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Fraud Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
for col in ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']:
    fraud_rate = df.groupby(col, observed=True)['fraud_bool'].mean() * 100
    print(f"\nFraud rate by {col} (%):")
    print(fraud_rate.round(2).sort_values(ascending=False))

## Preparing data for modelling
**No LabelEncoder.** Categorical columns stay as the `category` type, so their real names (e.g. employment status) remain readable for SHAP plots and the fairness audit:
- Logistic regression uses one-hot encoding inside its pipeline (LabelEncoder would impose a false ordering on the categories).
- LightGBM can use `category` columns directly in the next notebook.

**Protected attributes** (`customer_age`, `income`, `employment_status`) are currently kept as model features. Whether to keep them in or use them only for the audit is a methodological decision to justify in the dissertation.

In [ ]:
df = df.drop(columns=['age_group'])  # only needed for EDA

PROTECTED_ATTRIBUTES = ['customer_age', 'income', 'employment_status']
TARGET = 'fraud_bool'

## Train/test split
The BAF paper uses a **temporal split**: months 0–5 for training and months 6–7 for testing. This mimics deployment (train on the past, predict the future) and makes results comparable with the BAF benchmark. `month` is then removed from the features, because the model shouldn't learn from the calendar month itself.

Set `SPLIT_METHOD = 'random'` to use a stratified random 90/10 split instead (this choice would need justifying).

In [ ]:
from sklearn.model_selection import train_test_split

SPLIT_METHOD = 'temporal'   # 'temporal' (BAF paper) or 'random'

if SPLIT_METHOD == 'temporal':
    train_mask = df['month'] <= 5
    train_df, test_df = df[train_mask], df[~train_mask]
else:
    train_df, test_df = train_test_split(
        df, test_size=0.1, random_state=RANDOM_STATE, stratify=df[TARGET]
    )

FEATURES = [c for c in df.columns if c not in [TARGET, 'month']]
CAT_FEATURES = [c for c in categorical_cols if c in FEATURES]
NUM_FEATURES = [c for c in FEATURES if c not in CAT_FEATURES]

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

del train_df, test_df
gc.collect()

print(f"Split method: {SPLIT_METHOD}")
print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print(f"\nFraud rate in training set: {y_train.mean() * 100:.2f}%  ({y_train.sum()} cases)")
print(f"Fraud rate in test set:     {y_test.mean() * 100:.2f}%  ({y_test.sum()} cases)")

## Evaluation function
Fraud is ~1% of cases, so accuracy and AUC-ROC look optimistic. As set out in the proposal, models are compared on **AUC-PR, precision, recall and F1**. **Recall at 5% false-positive rate** is also reported because it is the headline metric in the BAF paper.

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, roc_curve, confusion_matrix,
)

def recall_at_fpr(y_true, y_prob, target_fpr=0.05):
    """Highest recall achievable while keeping the false-positive rate at or below target_fpr."""
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(tpr[fpr <= target_fpr].max())

def evaluate(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    print(f"--- {name} (threshold = {threshold}) ---")
    print(classification_report(y_true, y_pred, target_names=['Legitimate', 'Fraud'], digits=3))
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_true, y_pred))
    results = {
        'model': name,
        'AUC-ROC': roc_auc_score(y_true, y_prob),
        'AUC-PR': average_precision_score(y_true, y_prob),
        'Recall@5%FPR': recall_at_fpr(y_true, y_prob, 0.05),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
    }
    print({k: round(v, 4) for k, v in results.items() if k != 'model'})
    return results

all_results = []

## Baseline: logistic regression

In [ ]:
import time
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocess = ColumnTransformer([
    ('num', StandardScaler(), NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_FEATURES),
])

lr_model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(
        class_weight='balanced',   # handles the ~1% class imbalance
        max_iter=2000,
        random_state=RANDOM_STATE,
    )),
])

print("Training Logistic Regression (scaled numeric + one-hot categorical)...")
start = time.time()
lr_model.fit(X_train, y_train)
print(f"Training complete in {round(time.time() - start, 1)} seconds\n")

lr_prob = lr_model.predict_proba(X_test)[:, 1]
all_results.append(evaluate('Logistic Regression', y_test, lr_prob))

## Save outputs for the next notebooks

In [ ]:
import joblib

# Processed train/test sets (category types are preserved in Parquet)
X_train.to_parquet(OUTPUT_DIR / 'processed' / 'X_train.parquet')
X_test.to_parquet(OUTPUT_DIR / 'processed' / 'X_test.parquet')
y_train.to_frame().to_parquet(OUTPUT_DIR / 'processed' / 'y_train.parquet')
y_test.to_frame().to_parquet(OUTPUT_DIR / 'processed' / 'y_test.parquet')

# Trained model and feature lists
joblib.dump(lr_model, OUTPUT_DIR / 'models' / 'logistic_regression.joblib')
joblib.dump(
    {'FEATURES': FEATURES, 'CAT_FEATURES': CAT_FEATURES, 'NUM_FEATURES': NUM_FEATURES,
     'PROTECTED_ATTRIBUTES': PROTECTED_ATTRIBUTES, 'SPLIT_METHOD': SPLIT_METHOD},
    OUTPUT_DIR / 'models' / 'feature_config.joblib',
)

# Results table
results_df = pd.DataFrame(all_results).round(4)
results_df.to_csv(OUTPUT_DIR / 'results' / 'baseline_results.csv', index=False)
print(f"Saved outputs to {OUTPUT_DIR}")
results_df